# Research graph

## Input

In [2]:
RESULTS_PATH = './results'
SUBJECT = 'Ленинградская область'
DISTRICTS = {
    'boksitogorsk': 'Бокситогорский муниципальный район',
    'volosovo': 'Волосовский муниципальный район',
    'volhov': 'Волховский муниципальный район',
    'vsevolozhsk': 'Всеволожский муниципальный район',
    'vyborg': 'Выборгский муниципальный район',
    'kingisepp': 'Кингисеппский муниципальный район',
    'kirishi': 'Киришский муниципальный район',
    'kirovsk': 'Кировский муниципальный район',
    'lodeynoepole': 'Лодейнопольский муниципальный район',
    'lomonosov': 'Ломоносовский муниципальный район',
    'luga': 'Лужский муниципальный район',
    'podporozhye': 'Подпорожский муниципальный район',
    'priozersk': 'Приозерский муниципальный район',
    'slantsy': 'Сланцевский муниципальный район',
    'tihvin': 'Тихвинский муниципальный район',
    'tosno': 'Тосненский муниципальный район',
    'sosnovybor': 'Сосновоборский городской округ',
    'gatchina': 'Гатчинский муниципальный округ',
}
YEAR = 2026
PLAN = {
    'диагностика экономического и пространственного состояния МО': [
        'социально-экономический анализ (демография, производительность труда, структура и динамика ВГП, рынок труда, рынок жилья и коммерческой недвижимости, бюджетная обеспеченность)',
        'пространственный анализ (функциональная организация территории, структура землевладения, природноэкологический каркас, архитектурно-градостроительная среда)',
        'транспортный анализ (городской и внешний пассажирский и грузовой транспорт, парковки, пешеходные зоны)',
        'анализ инженерной инфраструктуры (водоснабжение и водоотведение, энергоснабжение, теплоснабжение)',
    ]
}

## Store

## Graphs

In [3]:
from src.tools import IndicatorsStore

store = IndicatorsStore('./data/indicators.xlsx')
indicators_tool = store.tool

### QA

In [4]:
from src.graphs.qa_agent import create_qa_graph

qa_graph = create_qa_graph()

### Research

In [5]:
from src.graphs.research import create_research_graph

research_graph = create_research_graph(qa_graph, 4)

### Goal

In [6]:
from src.graphs.goal import create_goal_graph

goal_graph = create_goal_graph()

## Experiments

In [7]:
from pydantic import BaseModel
import json

def write_json(data, file_path : str):
    with open(file_path, mode='w') as f:
        if isinstance(data, BaseModel):
            data = data.model_dump()
        json.dump(data, f)

def read_json(file_path : str) -> dict:
    with open(file_path, 'r') as f:
        return json.load(f)

In [13]:
import os
from tqdm import tqdm

def perform_init(path : str):
    if not os.path.exists(path):
        os.mkdir(path)

def perform_analysis(path : str, params : dict):
    file_path = os.path.join(path, 'analysis.json')
    if os.path.exists(file_path):
        result = read_json(file_path)
    else:
        state = research_graph.invoke({'params': params})
        result = state['result']
        write_json(result, file_path)
    return result

def perform_goal(path : str, analysis : dict):
    file_path = os.path.join(path, 'goal_setting.json')
    if os.path.exists(file_path):
        result = read_json(file_path)
    else:
        state = goal_graph.invoke({'analysis': analysis, 'max_iters': 3, 'tools': [indicators_tool]})
        result = state['result']
        write_json(result, file_path)
    return result

for district_code, district_name in tqdm(DISTRICTS.items()):
    path = os.path.join(RESULTS_PATH, district_code)
    params = {
        'location': f'{district_name} ({SUBJECT})',
        'year': YEAR,
        'plan': PLAN
    }

    perform_init(path)
    analysis = perform_analysis(path, params)
    goal = perform_goal(path, analysis)

100%|██████████| 18/18 [1:10:28<00:00, 234.91s/it]
